# Fábrica de Corpus FOMEZERO — doação segura de alimentos (PLN)
Global Solution 2026.2 · FIAP · Tecnólogo em IA · turma 2TIAPR.

Transforma documentos oficiais (D1 Lei nº 15.224/2025, D2 RDC ANVISA nº 216/2004, D3 Guia ANVISA nº 57/2022,
D4 Guia ANVISA de Boas Práticas para Bancos de Alimentos) em três corpus JSONL para fine-tuning de um SLM:
`corpus_texto.jsonl`, `corpus_qa.jsonl` e `corpus_chat.jsonl`.

Etapas (uma seção cada): 0 setup · 1 ingestão · 2 extração · 3 limpeza · 4 normalização · 5 anonimização ·
6 segmentação · 7 geração dos corpus (7a texto; 7b QA e 7c chat com IA + revisão humana) · 8 relatório de qualidade · 9 interface Gradio.

**Regras do projeto:** sem remoção de *stop words* e sem *stemming*; nenhum dado pessoal real nos corpus;
nenhuma resposta declara que um alimento é seguro para consumo (o atendente orienta e encaminha à vigilância
sanitária do município ou a um profissional habilitado). Todas as funções estão no pacote `fabrica/`; o script
`executar_pipeline.py` chama as mesmas funções.

## 0. Setup
**No Google Colab:** copie a pasta `05_ENTREGAVEIS/PLN` do repositório para o seu Google Drive (ex.: `MyDrive/GS_FOMEZERO/PLN`)
e ajuste `PASTA_PLN` abaixo. A célula monta o Drive e instala `requirements.txt`. Para a seção 7 (geração com IA)
use GPU: *Ambiente de execução > Alterar o tipo de ambiente > T4 GPU*.
**Local:** abra o notebook a partir da pasta `05_ENTREGAVEIS/PLN` com o ambiente já instalado (`pip install -r requirements.txt`).

In [1]:
import json, os, subprocess, sys
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_PLN = Path("/content/drive/MyDrive/GS_FOMEZERO/PLN")   # ajuste se copiou para outro lugar
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PASTA_PLN / "requirements.txt")], check=True)
else:
    PASTA_PLN = Path.cwd()

os.environ["FABRICA_PASTA_PLN"] = str(PASTA_PLN)
sys.path.insert(0, str(PASTA_PLN))
from fabrica import config
print("Python", sys.version.split()[0], "| pasta:", config.PASTA_PLN, "| Colab:", EM_COLAB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python 3.13.16 | pasta: /content/drive/MyDrive/GS_FOMEZERO/PLN | Colab: True


## 1. Ingestão (M1) — PDF, TXT, DOCX, HTML e texto colado [p.1 §2.1]
Lê `documentos/fontes/` (D1 a D4, bytes originais), confere a assinatura de cada arquivo, conta as páginas dos
PDFs e verifica se há de 3 a 5 documentos-fonte com pelo menos 20 páginas (contamos só páginas de PDF, DL-PLN-03).
No Colab, `USAR_UPLOAD = True` abre o seletor de arquivos; o texto colado entra como `texto_colado_NN` com a URL de origem.
Os arquivos `teste_SIMULADO.*` (TXT e DOCX) são só testes de formato: não entram nos corpus finais.

In [2]:
from fabrica.m1_ingestao import ingerir_bytes, ingerir_pasta, ingerir_texto_colado, verificar_conjunto

docs = ingerir_pasta(config.PASTA_FONTES)
verificacao = verificar_conjunto(docs)
for d in verificacao["documentos"]:
    print(d)
print("Total de páginas de PDF:", verificacao["total_paginas_pdf"], "| OK:", verificacao["ok"], verificacao["avisos"])

USAR_UPLOAD = False          # True no Colab para enviar outros arquivos (vão só para a demonstração)
if EM_COLAB and USAR_UPLOAD:
    from google.colab import files
    enviados = files.upload()
    extras = [ingerir_bytes(nome, dados, f"U{i:02d}") for i, (nome, dados) in enumerate(enviados.items(), 1)]
    print("Enviados:", [(e.fonte, e.formato, e.paginas) for e in extras])

from fabrica.pipeline import texto_colado_demonstracao
colado = ingerir_texto_colado(texto_colado_demonstracao(), 1, config.DOCUMENTOS_FONTE["D1_Lei_15224_2025.htm"]["url"])
print("Texto colado:", colado.fonte, "| origem:", colado.origem_url, "| início:", colado.texto[:120])

{'doc_id': 'D1', 'fonte': 'D1_Lei_15224_2025.htm', 'formato': 'html', 'tipo': 'fonte_oficial', 'paginas': None, 'bytes': 40986}
{'doc_id': 'D2', 'fonte': 'D2_RDC_ANVISA_216_2004.html', 'formato': 'html', 'tipo': 'fonte_oficial', 'paginas': None, 'bytes': 53535}
{'doc_id': 'D3', 'fonte': 'D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf', 'formato': 'pdf', 'tipo': 'fonte_oficial', 'paginas': 113, 'bytes': 3002587}
{'doc_id': 'D4', 'fonte': 'D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf', 'formato': 'pdf', 'tipo': 'fonte_oficial', 'paginas': 33, 'bytes': 830946}
Total de páginas de PDF: 146 | OK: True []
Texto colado: texto_colado_01 | origem: https://www.planalto.gov.br/ccivil_03/_ato2023-2026/2025/lei/l15224.htm | início: Art. 14. Poderão ser doados a bancos de alimentos e a instituições receptoras e diretamente aos beneficiários os aliment


## 2. Extração com origem (M2) [p.1 §2.2]
PDF por página (`pdfplumber`, `x_tolerance=1.5`; fallback `pypdf`), DOCX por parágrafo e tabela (`python-docx`),
HTML por bloco (`BeautifulSoup` + `lxml`, sem script/style; header/nav/footer marcados para a limpeza) e TXT/HTML
com detecção de codificação (a página do Planalto não declara a codificação: é lida como windows-1252).
Cada bloco guarda `fonte` e `pagina` (PDF) ou `secao`. Metadados de PDF não são lidos (DEC-011).

In [3]:
from fabrica.m2_extracao import extrair

blocos = {d.fonte: extrair(d) for d in docs}
for fonte, bs in blocos.items():
    b = bs[0]
    print(f"{fonte}: {len(bs)} blocos | 1º bloco pagina={b['pagina']} secao={b['secao']} | {b['texto'][:100]!r}")

D1_Lei_15224_2025.htm: 81 blocos | 1º bloco pagina=None secao=1 | 'Presidência da República\nCasa Civil\nSecretaria Especial para Assuntos Jurídicos'
D2_RDC_ANVISA_216_2004.html: 172 blocos | 1º bloco pagina=None secao=1 | 'ADVERTÊNCIA'
D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf: 113 blocos | 1º bloco pagina=1 secao=None | 'Guia para Doação de Alimentos com Segurança\nSanitária\nGuia nº 57/2022 – versão 1\nAgência Nacional de'
D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf: 33 blocos | 1º bloco pagina=1 secao=None | '1\no\nã\ns\nr\ne\nV Agência Nacional de Vigilância Sanitária\n–\n9\n1\n0\n2\n/\n6\n2\nº\nN\nA\nI\nU\nG\nGUIA DE BOAS PRÁT'


## 3. Limpeza (M3) [p.1 §2.3]
Regras (DESENVOLVIMENTO §2.3): cabeçalho/rodapé de PDF = 2 primeiras/últimas linhas repetidas em ≥ 50% das páginas;
HTML = header/nav/footer e blocos institucionais; numeração de página; URLs soltas (a frase fica); caracteres
estranhos (controles, U+FFFD, largura zero, uso privado, marcadores, pontilhado de sumário, texto vertical de capa);
linhas repetidas (seguidas ou título corrido); hifenização no fim da linha (mantém o hífen em pronome oblíquo e em
palavra composta que aparece com hífen no documento). Tudo vai para o log.

In [4]:
from fabrica.m3_limpeza import limpar_documento, resumir_log

limpos, log_limpeza = {}, []
for fonte, bs in blocos.items():
    limpos[fonte], lg = limpar_documento(bs)
    log_limpeza += lg
for fonte, cont in resumir_log(log_limpeza).items():
    print(fonte, cont)
print("Exemplos do log:", [ (e["regra"], e["removido"]) for e in log_limpeza if e["regra"] in ("cabecalho_pdf", "url_solta", "hifenizacao_juntada")][:6])

D1_Lei_15224_2025.htm {'bloco_institucional_html': 3}
D2_RDC_ANVISA_216_2004.html {'bloco_institucional_html': 4}
D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf {'numeracao_pagina': 113, 'caracteres_estranhos': 233, 'secao_referencias': 203, 'secao_referencias_fim': 1, 'url_solta': 1, 'url_solta_linha': 1, 'linha_repetida_titulo_corrido': 15, 'hifenizacao_mantida': 1, 'hifenizacao_juntada': 2}
D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf {'texto_vertical_diagramacao': 3, 'numeracao_pagina': 32, 'caracteres_estranhos': 60, 'cabecalho_pdf': 32, 'secao_referencias': 77, 'url_solta': 4, 'url_solta_linha': 1, 'resto_url_continuacao': 2, 'hifenizacao_mantida': 4}
Exemplos do log: [('url_solta', 'https://pesquisa.anvisa.gov.br/index.php/577264?lang=pt-BR'), ('hifenizacao_juntada', 'pós-|tratamento'), ('hifenizacao_juntada', 'vermelho-|cereja'), ('cabecalho_pdf', 'ALIMENTOS - GUIA nº 26, versão 1, de 21 de junho de 2019'), ('cabecalho_pdf', 'ALIMENTOS - GUIA nº 26, versão 1, de 21 de junh

## 4. Normalização (M4) [p.1 §2.4]
UTF-8 e Unicode NFC; espaços (NBSP e tabulação viram espaço; espaços repetidos colapsam); quebras de linha
(linhas do mesmo parágrafo se juntam; parágrafo que continua na página seguinte é unido); aspas curvas viram retas.
**Acentos, pontuação e maiúsculas são mantidos** — o teste abaixo compara as contagens antes e depois.

In [5]:
from fabrica.m4_normalizacao import normalizar_documento

paragrafos, testes_m4 = {}, {}
for fonte, ls in limpos.items():
    paragrafos[fonte], testes_m4[fonte] = normalizar_documento(ls)
    print(fonte, "| parágrafos:", len(paragrafos[fonte]), "| preservado:", testes_m4[fonte]["preservado"], testes_m4[fonte]["depois"])

D1_Lei_15224_2025.htm | parágrafos: 78 | preservado: True {'acentuadas': 440, 'maiusculas': 539, 'pontuacao': 328}
D2_RDC_ANVISA_216_2004.html | parágrafos: 174 | preservado: True {'acentuadas': 1413, 'maiusculas': 1410, 'pontuacao': 1027}
D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf | parágrafos: 751 | preservado: True {'acentuadas': 5908, 'maiusculas': 4385, 'pontuacao': 5125}
D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf | parágrafos: 200 | preservado: True {'acentuadas': 1715, 'maiusculas': 1620, 'pontuacao': 1200}


## 5. Anonimização (M5) [p.1 §2.5; p.3 §4.2]
Ordem fixa: e-mail → CPF → telefone → nomes de pessoas (NER `PER` do spaCy `pt_core_news_lg` 3.8.0, licença CC BY-SA 4.0;
regra do bloco de assinaturas do Planalto; autores de referências bibliográficas). Máscaras `[EMAIL]`, `[CPF]`,
`[TELEFONE]`, `[NOME]`. Qualquer formato de CPF e qualquer nome de pessoa (inclusive agentes públicos) são mascarados
(DL-PLN-11). Para reduzir falsos positivos do NER, uma entidade cuja maioria das palavras aparece em minúscula nos
próprios documentos (ex.: "Resíduos", "Segurança Alimentar") não é tratada como nome.
Abaixo, o exemplo **SIMULADO** de antes/depois (dados fictícios).

In [6]:
from fabrica.m5_anonimizacao import anonimizar_documento, definir_vocabulario
from fabrica.pipeline import exemplo_anonimizacao

definir_vocabulario(p["texto"] for ps in paragrafos.values() for p in ps)
anonimos, contagem = {}, {}
for fonte, ps in paragrafos.items():
    anonimos[fonte], _, contagem[fonte] = anonimizar_documento(ps)
    print(fonte, contagem[fonte])
ex = exemplo_anonimizacao()
print("ANTES (SIMULADO):", ex["antes_SIMULADO"])
print("DEPOIS          :", ex["depois"])

D1_Lei_15224_2025.htm {'EMAIL': 0, 'CPF': 0, 'TELEFONE': 0, 'NOME': 1}
D2_RDC_ANVISA_216_2004.html {'EMAIL': 0, 'CPF': 0, 'TELEFONE': 0, 'NOME': 3}
D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf {'EMAIL': 0, 'CPF': 0, 'TELEFONE': 0, 'NOME': 33}
D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf {'EMAIL': 0, 'CPF': 0, 'TELEFONE': 0, 'NOME': 3}
ANTES (SIMULADO): Contato do doador: Maria Fictícia da Silva, CPF 123.456.789-00, e-mail maria.ficticia@example.com, telefone (11) 90000-0000. O lote de doação sai às 14h.
DEPOIS          : Contato do doador: [NOME], CPF [CPF], e-mail [EMAIL], telefone [TELEFONE]. O lote de doação sai às 14h.


## 6. Segmentação (M6) [p.1 §2.6]
Trechos de **256 a 512 tokens** medidos com o tokenizador de `Qwen/Qwen2.5-7B-Instruct` (só os arquivos do tokenizador
são baixados, sem pesos e sem chave). Corte só em fim de frase ou de parágrafo; sobra < 256 é unida ou repartida;
documento inteiro < 256 tokens fica fora do corpus de texto e é informado. Só aceita texto anonimizado.

In [7]:
from fabrica.m6_segmentacao import segmentar_documento

trechos_demo = {}
for fonte, ps in anonimos.items():
    trechos_demo[fonte], info = segmentar_documento(ps)
    toks = [t["n_tokens"] for t in trechos_demo[fonte]]
    print(fonte, "| trechos:", len(toks), "| tokens mín/máx:", (min(toks), max(toks)) if toks else None, "| exceções:", info.get("excecoes"))

D1_Lei_15224_2025.htm | trechos: 7 | tokens mín/máx: (351, 510) | exceções: []
D2_RDC_ANVISA_216_2004.html | trechos: 22 | tokens mín/máx: (340, 512) | exceções: []
D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf | trechos: 93 | tokens mín/máx: (313, 512) | exceções: []
D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf | trechos: 27 | tokens mín/máx: (421, 512) | exceções: []


## 7. Geração dos três corpus (M7) [p.1 §2.7; p.2 §3]
### 7a. Texto puro (sem GPU)
Executa o pipeline completo (as mesmas funções das seções 1–6, mais a varredura de dados pessoais no nível do trecho)
e grava `corpus/corpus_texto.jsonl` (`{"text", "fonte"}`), `saidas/mapa_trechos.jsonl`
(`{"indice_linha", "fonte", "pagina_inicial", "pagina_final"}`, `indice_linha` começa em 0), os logs e as estatísticas.

In [8]:
from fabrica.pipeline import gerar_saidas, processar_documentos

resultado = processar_documentos(docs)
saidas = gerar_saidas(resultado, [config.PASTA_CORPUS], config.PASTA_SAIDAS)   # corpus_texto oficial só em corpus/
print(saidas["estatisticas"]["corpus_texto"])
print(json.dumps(saidas["registros"][0], ensure_ascii=False)[:400])

[M2] extraindo D1_Lei_15224_2025.htm (html)
[M3] limpando D1_Lei_15224_2025.htm: 81 blocos extraídos
[M4] normalizando D1_Lei_15224_2025.htm
[M2] extraindo D2_RDC_ANVISA_216_2004.html (html)
[M3] limpando D2_RDC_ANVISA_216_2004.html: 172 blocos extraídos
[M4] normalizando D2_RDC_ANVISA_216_2004.html
[M2] extraindo D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf (pdf)
[M3] limpando D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf: 113 blocos extraídos
[M4] normalizando D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf
[M2] extraindo D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf (pdf)
[M3] limpando D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf: 33 blocos extraídos
[M4] normalizando D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf
[M5] vocabulário de palavras em minúscula dos documentos (filtro de falso positivo do NER): 4223
[M5] anonimizando D1_Lei_15224_2025.htm: 78 parágrafos
[M6] segmentando D1_Lei_15224_2025.htm
[M5] anonimizando D2_RDC_ANVISA_216_2004.html: 174 parágrafos
[M6] s

### 7b e 7c. Perguntas e respostas, diálogos e recusas com IA (Colab + GPU) e revisão humana
**Modelo:** `Qwen/Qwen2.5-7B-Instruct` (revisão fixada em `config.REVISAO_QWEN`), 4 bits NF4 (bitsandbytes),
temperatura 0,3, top_p 0,9, semente 42 + nº do item. **Prompts:** `revisao/prompts/*.txt` (vão inteiros para o relatório).
Cada candidato passa por filtros automáticos: evidência copiada literalmente do trecho; nenhuma afirmação de que o
alimento é seguro; sem dado pessoal; sem termos do FOMEZERO (pontos, Selo Verde ESG, CO₂) que não estejam no documento;
sem citar a Lei 14.016 (revogada). Trechos do D3 que citam a Lei 14.016 não geram QA nem diálogo (DL-PLN-05).
As 12 recusas partem de `revisao/prompts/perguntas_recusa.json` (5 fora do tema + 7 encaminhamentos).

**Como o usuário executa no Colab:**
1. Ambiente com GPU T4; rode as seções 0 a 7a.
2. Mude `REGERAR = True` e rode a célula abaixo (≈ 1 a 2 horas; se a sessão cair, rode de novo: a geração continua de onde parou).
3. Saem em `revisao/`: `qa_candidatos.jsonl`, `chat_candidatos.jsonl`, `log_geracao.jsonl`, `revisao_qa.csv`, `revisao_chat.csv`.
4. O revisor (Leonardo Melo, RM 567097) preenche nos CSV (separador `;`): `data_revisao`, C1 a C7 (`sim`/`nao`),
   `decisao` (`aprovado`/`corrigido`/`rejeitado`) e, se corrigido, `correcao` (QA: novo texto da resposta;
   chat: JSON com a lista de mensagens user/assistant). Revisão: 25% de cada corpus (estratificado por documento)
   + 100% das recusas e dos itens duvidosos.
5. Devolva os arquivos de `revisao/` para a pasta `05_ENTREGAVEIS/PLN/revisao/` do repositório (no Drive eles já ficam lá)
   e rode a célula seguinte (`aplicar_revisao`), que grava `corpus/corpus_qa.jsonl` e `corpus/corpus_chat.jsonl`.

In [9]:
from fabrica import m7_ia
from fabrica.m7_texto import ler_jsonl

REGERAR = True   # True só no Colab com GPU, para chamar o modelo
registros_texto = ler_jsonl(config.PASTA_CORPUS / config.ARQ_CORPUS_TEXTO)
if REGERAR:
    m7_ia.registrar_ambiente()
    modelo, tok = m7_ia.carregar_modelo()
    m7_ia.gerar_candidatos_qa(modelo, tok, registros_texto)
    m7_ia.gerar_candidatos_dialogo(modelo, tok, registros_texto)
    m7_ia.gerar_recusas(modelo, tok, registros_texto)
    m7_ia.preparar_revisao(registros_texto)
else:
    for n in ("qa_candidatos.jsonl", "chat_candidatos.jsonl", "revisao_qa.csv", "revisao_chat.csv"):
        print(n, "existe" if (config.PASTA_REVISAO / n).exists() else "AINDA NÃO GERADO (rode no Colab com REGERAR = True)")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[7c] recusa R06 tentativa 4: rejeitado_filtro
[7c] recusa R06 tentativa 5: rejeitado_filtro
[7c] recusa R06 tentativa 6: aprovado_filtro
[7c] recusa R07 tentativa 4: aprovado_filtro
[7c] recusa R08 tentativa 4: rejeitado_filtro
[7c] recusa R08 tentativa 5: aprovado_filtro
[7c] recusa R12 tentativa 4: rejeitado_filtro
[7c] recusa R12 tentativa 5: aprovado_filtro
[revisão] qa: {'candidatos': 172, 'rejeitados_filtro': 29, 'duplicatas_removidas': 0, 'base_ia': 143, 'na_revisao': 41, 'por_motivo': {'sorteio_25pct_estratificado': 35, 'duvidoso': 6}}
[revisão] chat: {'candidatos': 184, 'rejeitados_filtro': 123, 'duplicatas_removidas': 0, 'base_ia': 52, 'na_revisao': 29, 'por_motivo': {'sorteio_25pct_estratificado': 9, 'duvidoso': 8, 'recusa_100pct': 12}}


In [10]:
# Aplica a revisão humana (só funciona depois que os CSV estiverem preenchidos)
estat_geracao = None
try:
    estat_geracao = m7_ia.aplicar_revisao(registros_texto)
except m7_ia.RevisaoIncompleta as e:
    print("Revisão ainda não aplicada:", e)

Revisão ainda não aplicada: qa: revisão incompleta (123 problemas). Primeiros: ['QA-0003: falta revisor ou data', 'QA-0003: C1 a C7 precisam ser sim/nao', 'QA-0003: decisão precisa ser aprovado/corrigido/rejeitado', 'QA-0006: falta revisor ou data', 'QA-0006: C1 a C7 precisam ser sim/nao']


## 8. Relatório de qualidade (M8) [p.2 §2.8]
Número de exemplos, média de tokens por exemplo, duplicatas removidas e cinco exemplos aleatórios (semente 42) de
cada corpus, mais as validações (formato, `fonte`, mínimos 50/100/30, ≥ 2 trocas, ≥ 10 recusas, 256–512 tokens,
varredura de dados pessoais = 0, acentos preservados e o teste "sem stop words / sem stemming").
Corpus ainda não gerado aparece como PENDENTE. Também é gravado em `saidas/relatorio_qualidade.md`.

In [11]:
from fabrica.m8_qualidade import gerar_relatorio

eg_arq = config.PASTA_SAIDAS / "estatisticas_geracao.json"
eg = json.loads(eg_arq.read_text(encoding="utf-8")) if eg_arq.exists() else None
relatorio_md, relatorio = gerar_relatorio(config.PASTA_CORPUS, saidas["estatisticas"], eg)
(config.PASTA_SAIDAS / "relatorio_qualidade.md").write_text(relatorio_md, encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(relatorio_md))
except ImportError:
    print(relatorio_md[:3000])

# Relatório de qualidade — Fábrica de Corpus FOMEZERO (PLN)

Gerado em 2026-11-08T22:32:31 por `fabrica.m8_qualidade.gerar_relatorio` (números da execução real).
Tokens contados com o tokenizador de `Qwen/Qwen2.5-7B-Instruct` (sem tokens especiais). Texto: `text`; QA: `instruction`+`input`+`output`; chat: soma dos `content`. Exemplos: `random.Random(42)`.

## Estatísticas por corpus

| Corpus | Arquivo | Status | Nº de exemplos | Média de tokens | Mín–máx | Duplicatas removidas |
|---|---|---|---|---|---|---|
| texto | `corpus_texto.jsonl` | OK | 149 | 477.1 | 313–512 | 0 |
| qa | `corpus_qa.jsonl` | PENDENTE (arquivo ainda não gerado; geração no Colab) | — | — | — | — |
| chat | `corpus_chat.jsonl` | PENDENTE (arquivo ainda não gerado; geração no Colab) | — | — | — | — |

## Validações gerais

| Validação | Resultado | Detalhe |
|---|---|---|
| acentos, pontuação e maiúsculas preservados no M4 (todos os documentos) | OK | D1_Lei_15224_2025.htm: {'acentuadas': 440, 'maiusculas': 539, 'pontuacao': 328}; D2_RDC_ANVISA_216_2004.html: {'acentuadas': 1413, 'maiusculas': 1410, 'pontuacao': 1027}; D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf: {'acentuadas': 5908, 'maiusculas': 4385, 'pontuacao': 5125}; D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf: {'acentuadas': 1715, 'maiusculas': 1620, 'pontuacao': 1200} |
| sem stop words removidas e sem stemming: mesma sequência de palavras M5 -> trechos (§2.2) | OK | D1_Lei_15224_2025.htm: 1789 palavras; D2_RDC_ANVISA_216_2004.html: 5697 palavras; D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf: 24206 palavras; D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf: 7097 palavras |
| nenhum módulo do NLTK (stemmer/lista de stop words) carregado | OK | [] |
| 3 a 5 documentos-fonte e >= 20 páginas (só PDF) | OK | 4 documentos; 146 páginas de PDF |

## Validações — texto

| Validação | Resultado | Detalhe |
|---|---|---|
| uma linha = um objeto JSON | OK | 149 linhas lidas com json.loads |
| chaves exatamente ['fonte', 'text'] | OK | 0 linhas fora do formato |
| campo fonte preenchido | OK | 0 linhas sem fonte |
| todos os trechos com 256 a 512 tokens | OK | 0 fora da faixa; mín 313, máx 512 |
| mínimo de 50 trechos | OK | 149 trechos |
| sem URL, resto de URL nem lista de referências (DEC-017) | OK | {"http": 0, "www.": 0, "?x=": 0, "_file.pdf": 0, "Disponível em:": 0} |
| varredura de dados pessoais = 0 | OK | {"EMAIL": 0, "CPF": 0, "TELEFONE": 0, "NOME": 0} |

## Por documento (corpus de texto)

| Fonte | Formato | Páginas | Blocos extraídos | Parágrafos | Tokens | Trechos | Máscaras (EMAIL/CPF/TEL/NOME) |
|---|---|---|---|---|---|---|---|
| D1_Lei_15224_2025.htm | html | não paginado | 81 | 78 | 3276 | 7 | 0/0/0/1 |
| D2_RDC_ANVISA_216_2004.html | html | não paginado | 172 | 174 | 10600 | 22 | 0/0/0/3 |
| D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf | pdf | 113 | 113 | 751 | 44301 | 93 | 0/0/0/33 |
| D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf | pdf | 33 | 33 | 200 | 12908 | 27 | 0/0/0/3 |

Limpeza (contagem por regra; detalhes em `log_limpeza.json`):

- `D1_Lei_15224_2025.htm`: {"bloco_institucional_html": 3}
- `D2_RDC_ANVISA_216_2004.html`: {"bloco_institucional_html": 4}
- `D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf`: {"numeracao_pagina": 113, "caracteres_estranhos": 233, "secao_referencias": 203, "secao_referencias_fim": 1, "url_solta": 1, "url_solta_linha": 1, "linha_repetida_titulo_corrido": 15, "hifenizacao_mantida": 1, "hifenizacao_juntada": 2}
- `D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf`: {"texto_vertical_diagramacao": 3, "numeracao_pagina": 32, "caracteres_estranhos": 60, "cabecalho_pdf": 32, "secao_referencias": 77, "url_solta": 4, "url_solta_linha": 1, "resto_url_continuacao": 2, "hifenizacao_mantida": 4}

Palavras gramaticais contadas em cada etapa (prova de que não há remoção de stop words):

- `D1_Lei_15224_2025.htm`: M2 539 · M5 530 · trechos 530 (a diferença M2→M5 vem só das remoções registradas no log de limpeza)
- `D2_RDC_ANVISA_216_2004.html`: M2 1512 · M5 1504 · trechos 1504 (a diferença M2→M5 vem só das remoções registradas no log de limpeza)
- `D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf`: M2 7042 · M5 6479 · trechos 6479 (a diferença M2→M5 vem só das remoções registradas no log de limpeza)
- `D4_Guia_ANVISA_Boas_Praticas_Bancos_Alimentos.pdf`: M2 2304 · M5 1996 · trechos 1996 (a diferença M2→M5 vem só das remoções registradas no log de limpeza)

## Cinco exemplos aleatórios — texto

Linha 28:

```json
{"text": "4.11.8 Os POP relacionados à higiene e saúde dos manipuladores devem contemplar as etapas, a freqüência e os princípios ativos usados na lavagem e anti-sepsia das mãos dos manipuladores, assim como as medidas adotadas nos casos em que os manipuladores apresentem lesão nas mãos, sintomas de enfermidade ou suspeita de problema de saúde que possa comprometer a qualidade higiênicosanitária dos alimentos. Deve-se especificar os exames aos quais os manipuladores de alimentos são submetidos, bem como a periodicidade de sua execução. O programa de capacitação dos manipuladores em higiene deve ser descrito, sendo determinada a carga horária, o conteúdo programático e a freqüência de sua realização, mantendo-se em arquivo os registros da participação nominal dos funcionários.\n\n4.12. RESPONSABILIDADE\n\n4.12.1. O responsável pelas atividades de manipulação dos alimentos deve ser o proprietário ou funcionário designado, devidamente capacitado, sem prejuízo dos casos onde há previsão legal para responsabilidade técnica.\n\n4.12.2. O responsável pelas atividades de manipulação dos alimentos deve ser comprovadamente submetido a curso de capacitação, abordando, no mínimo, os seguintes temas:\n\na) Contaminantes alimentares;\n\nb) Doenças transmitidas por alimentos;\n\nc) Manipulação higiênica dos alimentos;\n\nd) Boas Práticas.", "fonte": "D2_RDC_ANVISA_216_2004.html"}
```

Linha 6:

```json
{"text": "A doação de alimentos, nos termos desta Lei, constituirá exceção ao regime da responsabilidade objetiva disposto no art. 931 da Lei nº 10.406, de 10 de janeiro de 2002 (Código Civil), e nos arts. 12 e 13 da Lei nº 8.078, de 11 de setembro de 1990 (Código de Defesa do Consumidor).\n\nArt. 16. O doador de alimentos e o intermediário apenas responderão civilmente por danos ocasionados pelos alimentos doados quando houver dolo, nos termos do art. 392 da Lei nº 10.406, de 10 de janeiro de 2002 (Código Civil).\n\nArt. 17. A doação de alimentos, nos termos desta Lei, não configurará, em nenhuma hipótese, relação de consumo, ainda que haja finalidade de publicidade direta ou indireta.\n\nCAPÍTULO VI DISPOSIÇÕES FINAIS\n\nArt. 18. (VETADO).\n\nArt. 19. Fica revogada a Lei nº 14.016, de 23 de junho de 2020.\n\nArt. 20. Esta Lei entra em vigor na data de sua publicação.\n\nBrasília, 30 de setembro de 2025; 204º da Independência e 137º da República.\n\n[NOME]\n\n*", "fonte": "D1_Lei_15224_2025.htm"}
```

Linha 70:

```json
{"text": "O alimento acondicionado em lata não pode ser considerado seguro se a embalagem apresentar:\n\nAmassado na junção da costura lateral e na junção das extremidades (tampa e fundo);\n\nAlimentos • Amassado formando um dente - deformação pontuda; acondicionados em lata • Inchamento ou estufado;\n\nFerrugem ou vazamento;\n\nAusência de rótulo ou rótulo danificado impedindo identificação da designação do alimento, os dados do fabricante ou importador do produto, a data de validade e o lote.\n\nOs alimentos acondicionados em frascos de vidro não podem ser considerados seguros se a embalagem apresentar:\n\nFita de vedação interna ou resistente à violação faltando ou quebrada;\n\nSujeira e/ou vazamento por baixo da borda da tampa;\n\nAlimentos Tampa curvada, botão de vácuo levantado ou outra evidência de acondicionados em que a tampa foi aberta; frascos de vidro Tampas enferrujadas ou amassadas Rachadura ou trincado;\n\nAusência de rótulo ou rótulo danificado impedindo identificação da designação do alimento, os dados do fabricante ou importador do produto, a data de validade e o lote.\n\nFonte: Guia de Avaliação de alimentos doados aos bancos de alimentos, Ministério do Desenvolvimento Social, 2018.\n\n5.2 Expedição de alimentos doados Na expedição das doações aos intermediários ou beneficiários as principais recomendações referem-se à identificação e conferência de acondicionamento, pois os alimentos não podem estar expostos ao ambiente e sem proteção.\n\nPara realizar a expedição da doação, recomenda-se que:\n\nOs alimentos in natura, minimamente processados e refeições prontas para consumo estejam embalados adequadamente para o transporte, com a utilização de filme ou sacos plásticos novos e apropriados para uso alimentício ou bem acondicionados em caixas plásticas ou recipientes vedados e etiquetados conforme modelo apresentado a seguir:", "fonte": "D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf"}
```

Linha 62:

```json
{"text": "No caso de carnes, é importante que a mesma não apresente sinais de descongelamento/congelamento, como por exemplo: amolecimento da carne, embalagens úmidas e/ou molhada e acúmulo de cristais de gelo.\n\nO Anexo III deste Guia apresenta informações e imagens com exemplos de falhas na integridade em produtos cárneos, laticínios e ovos.\n\n5.1.4 Prazo de validade Conforme mencionado anteriormente, e de acordo com o Art. 1º da Lei de Doação, somente podem ser doados alimentos dentro do prazo de validade. Essa regra está alinhada ao Código de Defesa do Consumidor (Lei nº 8.078 de 1990) que considera impróprios para o consumo os alimentos cujos prazos de validade estejam vencidos (BRASIL, 1990).\n\nProdutos que não cumprem esses critérios precisam ter outros destinos, como alimentação animal, compostagem ou descarte adequado. Recomenda-se que os produtores de alimentos busquem outras estratégias antes de gerar desperdício e resíduo. Os prazos de validade seguem as legislações específicas, de acordo com o tipo de alimento (in natura e/ou processado). De acordo com o Guia nº 16/2018 da Anvisa, o prazo de validade inicia a partir do momento em que o alimento é preparado e/ou fabricado e precisa ser informado no rótulo dos alimentos embalados, observando as regras para a declaração das datas impostas pela RDC nº 727/2022, considerando que o estabelecimento do prazo de validade é de responsabilidade do fabricante.", "fonte": "D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf"}
```

Linha 57:

```json
{"text": "Salienta-se que nestes casos, as doações para lactentes podem ocorrer desde que cumpridas as regras da NBCAL.\n\nEntretanto, caso seja identificado qualquer episódio na cadeia produtiva e distributiva que possa afetar a qualidade nutricional do produto, recomenda-se que esse não seja doado, a não ser que o doador garanta por meio de análises laboratoriais a integridade nutricional do alimento.\n\n5.1.2 Segurança sanitária Na avaliação do alimento para doação, recomenda-se que o doador revise os aspectos que envolvem as Boas Práticas aos quais o alimento foi submetido durante o processamento, incluindo as planilhas de controle de temperatura e procedência das matérias primas, ingredientes e embalagens utilizadas. A segurança sanitária somente é garantida se os critérios de Boas Práticas forem cumpridos em todas as etapas de produção e manipulação do alimento, tanto na organização doadora, como nos grupos de voluntários que preparam um alimento para doação.\n\nNo caso das indústrias de alimentos e serviços de alimentação, vale reforçar que a segurança sanitária será assegurada com o cumprimento das legislações sanitárias federais, estaduais, municipais e distrital.\n\n5.1.3 Aspectos referentes a integridade dos alimentos Recomenda-se uma avaliação por meio de observação visual dos aspectos referentes a integridade dos alimentos. Falhas associadas à integridade dos alimentos podem levar diretamente a DTHA, sendo a terceira causa desse tipo de doença no Brasil.\n\nA avaliação verifica os aspectos sensoriais que compõe a integridade do alimento, como por exemplo: se a aparência geral, como cor e texturas, está com características aceitáveis, os aspectos da embalagem, conforme discutido no item 5.1.6 Condições de embalagem, entre outros. Somente podem ser doados alimentos aprovados na avaliação da integridade.", "fonte": "D3_Guia_ANVISA_57_2022_Doacao_Alimentos_v1.pdf"}
```



## 9. Interface Gradio
Abas: **Entrada** (upload múltiplo + texto colado), **Limpeza e anonimização** (antes/depois e exemplo SIMULADO),
**Corpus** (gera e baixa os arquivos) e **Qualidade** (relatório do M8). As saídas da interface vão para
`saidas/interface/<data-hora>/` e não substituem os corpus finais. No Colab o link é temporário (`share=True`).

In [12]:
from fabrica.interface import construir_interface

demo = construir_interface()
INICIAR_GRADIO = EM_COLAB      # localmente, mude para True se quiser abrir no navegador
if INICIAR_GRADIO:
    demo.launch(share=EM_COLAB)
else:
    print("Interface construída (", type(demo).__name__, "); launch() não chamado.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://827a9817ab304f63b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
